# Existence equation — panel logit

Reload every parquet written under `modeling/` into one pooled panel, lag the predetermined
regressors, and fit the pool-pair fixed-effects logit for whether a cross-pool arbitrage exists:

$$\Pr\!\big(D_{p,t}=1 \mid X_{p,t-1}\big)=\Lambda(\eta_{p,t}),\qquad \Lambda(z)=\frac{1}{1+e^{-z}}$$

$$
\eta_{p,t}=\alpha_p
+\beta_1\,\mathrm{Gap}_{p,t-1}
+\beta_2\,\log(\text{base\_fee}_t)
+\beta_3\,\text{gas\_util}_{t-1}
+\beta_4\,\log(1+\text{tip\_p90}_{t-1})
+\beta_5\,\overline{\log(1+\text{mev})}_{p,t-1}
+\beta_6\,\overline{\text{nb\_swaps\_ewma}}_{p,t-1}
+\beta_7\,\log(\text{ewma\_vol}_t)
+\beta_8\,\overline{\Delta\log L}_{p,t-1}
+\gamma_{h(t)}+\delta_{d(t)}+\eta\,\text{week}(t)
$$

`base_fee` and `ewma_vol` enter contemporaneously at `t`; every other regressor is lagged one
block to be predetermined. $\alpha_p$ are pool-pair fixed effects. All the assembly / estimation
logic lives in `arblib.modeling` — this notebook only calls it.

In [11]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd

from arblib import modeling
from arblib.config import STUDY as S
from arblib.modeling import EXISTENCE_TERMS

panel = modeling.build_panel(
    S.dependent_var_dir, S.pair_covariates_dir, S.chain_covariates_path, S.cex_vol_path
)
print("panel:", panel.shape)
panel.head(10)

panel: (4725, 20)


,pair,evt_block_time,evt_block_number,gap_q20,gap_q40,gap_q60,gap_q80,D_q20,D_q40,D_q60,D_q80,mev_intensity,frequency_intensity,per_log_liquidity_growth_rate_avg_venue,log_base_fee_per_gas,gas_util,log1p_tip_p50,log1p_tip_p90,vol_minute,ewma_vol
0,pancake_1_vs_pancake_2,2025-12-31 15:15:11+00:00,24133438,0.000000,0.0,0.0,0.0,0,0,0,0,22.918426,0.395834,NaN,18.842422,0.559332,17.952118,21.416413,2025-12-31 15:14:00+00:00,0.001042
1,pancake_1_vs_pancake_2,2025-12-31 15:15:23+00:00,24133439,2.282228,0.0,0.0,0.0,1,0,0,0,22.859350,0.434799,0.000000,18.857146,0.455957,18.520859,21.416413,2025-12-31 15:14:00+00:00,0.001042
2,pancake_1_vs_pancake_2,2025-12-31 15:15:35+00:00,24133440,2.282228,0.0,0.0,0.0,1,0,0,0,22.792684,0.406757,0.000000,18.846074,0.146020,17.901344,21.111583,2025-12-31 15:14:00+00:00,0.001042
3,pancake_1_vs_pancake_2,2025-12-31 15:15:47+00:00,24133441,0.000000,0.0,0.0,0.0,0,0,0,0,22.726029,0.412771,0.000000,18.753416,0.941608,16.462380,21.399657,2025-12-31 15:14:00+00:00,0.001042
4,pancake_1_vs_pancake_2,2025-12-31 15:15:59+00:00,24133442,0.000000,0.0,0.0,0.0,0,0,0,0,22.659559,0.418396,0.003868,18.858138,0.390697,15.437968,21.064384,2025-12-31 15:14:00+00:00,0.001042
5,pancake_1_vs_pancake_2,2025-12-31 15:16:11+00:00,24133443,0.000000,0.0,0.0,0.0,0,0,0,0,22.623704,0.520399,0.000000,18.830432,0.851834,17.327172,21.284515,2025-12-31 15:15:00+00:00,0.001030
6,pancake_1_vs_pancake_2,2025-12-31 15:16:23+00:00,24133444,0.000000,0.0,0.0,0.0,0,0,0,0,22.561954,0.519083,-0.003868,18.914735,0.515774,17.373660,21.416413,2025-12-31 15:15:00+00:00,0.001030
7,pancake_1_vs_pancake_2,2025-12-31 15:16:35+00:00,24133445,0.000000,0.0,0.0,0.0,0,0,0,0,22.495288,0.517852,0.000000,18.918671,0.335130,17.189564,21.353710,2025-12-31 15:15:00+00:00,0.001030
8,pancake_1_vs_pancake_2,2025-12-31 15:16:47+00:00,24133446,0.000000,0.0,0.0,0.0,0,0,0,0,22.434506,0.613441,0.000000,18.876580,0.464820,17.741907,21.416413,2025-12-31 15:15:00+00:00,0.001030
9,pancake_1_vs_pancake_2,2025-12-31 15:16:59+00:00,24133447,0.000000,0.0,0.0,0.0,0,0,0,0,22.367840,0.573878,0.000000,18.867746,0.605504,18.060541,21.416413,2025-12-31 15:15:00+00:00,0.001030


## Volatility join — no look-ahead

The EWMA volatility is a 1-minute series, but the panel's time unit is the block. A minute bar
labelled `T` only closes at `T+1min`, so each block uses the last **fully-closed** bar strictly
before its own minute: `vol_minute = floor(evt_block_time, 'min') − 1 min`. Blocks in minute
`15:15` are matched to the `15:14:00` vol, blocks in `15:16` to `15:15:00`, and so on — never a
bar that is still forming.

In [2]:
# panel.loc[panel.pair == "uniswap_1_vs_pancake_1",
#           ["evt_block_time", "vol_minute", "ewma_vol"]].head(10)

## Design matrix — lag the predetermined regressors

`prepare_model_frame` picks one trade-size quantile (its `D` is the outcome, its `Gap` the
regressor `gap_lag`), lags each pair-specific / `t-1` term one block **within** pair, keeps
`log(base_fee)` and `log(ewma_vol)` at `t`, and derives `hour` / `dow` / `week` for the calendar
fixed effects. The first block of every pair (and the second, for the liquidity-growth lag) drops
out; pool pairs whose `D` never varies in-sample are dropped (uninformative under pair FE, and
they would cause perfect separation). Set `quantile` to 0.2 / 0.4 / 0.6 / 0.8 for the four
reference trade sizes — 0.2 has by far the most existence events in this window.

In [3]:
model_df_closure = modeling.build_risk_set(panel, quantile=0.2, condition="closure")
model_df_onset   = modeling.build_risk_set(panel, quantile=0.2, condition="onset")

q20: dropping 10 pool pairs with no D variation (uninformative under pair FE): ['uniswap_2_vs_pancake_2', 'uniswap_2_vs_uniswap_3', 'uniswap_2_vs_uniswap_4', 'uniswap_3_vs_pancake_1', 'uniswap_3_vs_pancake_2', 'uniswap_3_vs_uniswap_4', 'uniswap_3_vs_uniswap_5', 'uniswap_4_vs_pancake_1', 'uniswap_4_vs_pancake_2', 'uniswap_4_vs_uniswap_5']
closure: dropping 4 pairs with no D variation: ['uniswap_1_vs_pancake_2', 'uniswap_1_vs_uniswap_3', 'uniswap_1_vs_uniswap_4', 'uniswap_2_vs_uniswap_5']
closure: (225, 15) | D mean: 0.5378 | pairs: 7
q20: dropping 10 pool pairs with no D variation (uninformative under pair FE): ['uniswap_2_vs_pancake_2', 'uniswap_2_vs_uniswap_3', 'uniswap_2_vs_uniswap_4', 'uniswap_3_vs_pancake_1', 'uniswap_3_vs_pancake_2', 'uniswap_3_vs_uniswap_4', 'uniswap_3_vs_uniswap_5', 'uniswap_4_vs_pancake_1', 'uniswap_4_vs_pancake_2', 'uniswap_4_vs_uniswap_5']
onset: (2216, 15) | D mean: 0.0501 | pairs: 11


## Fit the existence logit

Pooled logit with pool-pair fixed effects `C(pair)` and cluster-robust standard errors by pool
pair. Calendar fixed effects (`hour` / `dow` / `week`) are added only when the sample spans more
than one level — this single-hour extract has one of each, so they are omitted (they would be
collinear with the intercept). On a longer, multi-week run they enter automatically.

In [12]:
CLOSURE_TERMS = EXISTENCE_TERMS                                   # includes gap_lag
ONSET_TERMS = [t for t in EXISTENCE_TERMS if t != "gap_lag"]       # excludes it


res_onset_logit   = modeling.fit_hazard_logit(model_df_onset,   ONSET_TERMS,   cluster="pair", direction="logit")
res_onset_probit   = modeling.fit_hazard_logit(model_df_onset,   ONSET_TERMS,   cluster="pair", direction="probit")

res_closure_logit   = modeling.fit_hazard_logit(model_df_closure,   CLOSURE_TERMS,   cluster="pair", direction="logit")
res_closure_probit   = modeling.fit_hazard_logit(model_df_closure,   CLOSURE_TERMS,   cluster="pair", direction="probit")

In [10]:
print(res_onset_logit.summary())
print(res_onset_probit.summary())

                           Logit Regression Results                           
Dep. Variable:                      D   No. Observations:                 2216
Model:                          Logit   Df Residuals:                     2198
Method:                           MLE   Df Model:                           17
Date:                Fri, 24 Jul 2026   Pseudo R-squ.:                  0.1877
Time:                        14:15:19   Log-Likelihood:                -357.81
converged:                       True   LL-Null:                       -440.50
Covariance Type:              cluster   LLR p-value:                 2.310e-26
                                        coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------
Intercept                           116.0641     33.353      3.480      0.001      50.694     181.434
C(pair)[T.uniswap_1_vs_pancake_1]     0.8045      0.130      6

In [13]:
print(res_closure_logit.summary())
print(res_closure_probit.summary())

                           Logit Regression Results                           
Dep. Variable:                      D   No. Observations:                  225
Model:                          Logit   Df Residuals:                      210
Method:                           MLE   Df Model:                           14
Date:                Fri, 24 Jul 2026   Pseudo R-squ.:                 0.07325
Time:                        16:18:50   Log-Likelihood:                -143.94
converged:                       True   LL-Null:                       -155.32
Covariance Type:              cluster   LLR p-value:                   0.06437
                                        coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------
Intercept                           -17.4695     16.948     -1.031      0.303     -50.687      15.748
C(pair)[T.uniswap_1_vs_pancake_1]     0.5570      0.180      3

In [ ]:
corr_matrix = model_df_onset[ONSET_TERMS].corr()
corr_matrix

,log_base_fee,gas_util_lag,tip_p90_lag,mev_lag,freq_lag,log_vol,dlogL_lag
log_base_fee,1.000000,0.263105,0.027707,0.254081,0.165377,0.606667,-0.025433
gas_util_lag,0.263105,1.000000,0.151920,0.014608,0.036422,0.021402,-0.015902
tip_p90_lag,0.027707,0.151920,1.000000,-0.016915,0.007593,-0.007182,0.019422
mev_lag,0.254081,0.014608,-0.016915,1.000000,0.610371,0.343909,-0.001449
freq_lag,0.165377,0.036422,0.007593,0.610371,1.000000,0.166908,-0.000061
log_vol,0.606667,0.021402,-0.007182,0.343909,0.166908,1.000000,-0.001662
dlogL_lag,-0.025433,-0.015902,0.019422,-0.001449,-0.000061,-0.001662,1.000000


In [ ]:
print(model_df_onset[ONSET_TERMS].describe())


       log_base_fee  gas_util_lag  tip_p90_lag      mev_lag     freq_lag  \
count   2216.000000   2216.000000  2216.000000  2216.000000  2216.000000   
mean      18.691543      0.501234    21.181524    19.856562     0.638631   
std        0.108885      0.219935     1.443747     1.661244     0.425334   
min       18.419791      0.034508     1.098612    12.954298     0.058985   
25%       18.602025      0.356836    21.289997    19.194801     0.289487   
50%       18.694711      0.477841    21.414505    20.115109     0.537725   
75%       18.772161      0.639713    21.416413    20.892000     0.938172   
max       18.918671      0.999703    22.304658    23.065291     2.122847   

           log_vol    dlogL_lag  
count  2216.000000  2216.000000  
mean     -7.000576     0.000076  
std       0.078578     0.084330  
min      -7.168104    -1.618528  
25%      -7.067153     0.000000  
50%      -6.983427     0.000000  
75%      -6.940108     0.000000  
max      -6.866782     1.618528  


In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm

X = model_df_onset[["log_base_fee","gas_util_lag","tip_p90_lag","mev_lag","freq_lag","log_vol","dlogL_lag"]]
X = sm.add_constant(X)
vif = pd.Series([variance_inflation_factor(X.values, i) for i in range(X.shape[1])], index=X.columns)
print(vif)

const           98476.657605
log_base_fee        1.768493
gas_util_lag        1.135467
tip_p90_lag         1.024887
mev_lag             1.765243
freq_lag            1.605363
log_vol             1.746753
dlogL_lag           1.001477
dtype: float64


## Average marginal effects

Raw logit coefficients are not probability changes (unlike an LPM), so report AMEs — the sample
average of $\partial\Pr(D=1)/\partial X_k$ — with the clustered SEs, for the eight economic
covariates.

In [ ]:
modeling.average_marginal_effects(res_onset_logit, modeling.EXISTENCE_TERMS)

,dy/dx,Std. Err.,z,Pr(>|z|),Conf. Int. Low,Cont. Int. Hi.
log_base_fee,-0.167679,0.056628,-2.961063,0.003066,-0.278668,-0.056690
gas_util_lag,-0.035701,0.015919,-2.242630,0.024921,-0.066903,-0.004500
tip_p90_lag,-0.000420,0.001742,-0.241152,0.809438,-0.003835,0.002995
mev_lag,-0.004810,0.007696,-0.624969,0.531992,-0.019894,0.010274
freq_lag,0.070602,0.015386,4.588828,0.000004,0.040447,0.100757
log_vol,0.260106,0.140980,1.844979,0.065041,-0.016211,0.536422
dlogL_lag,-0.014041,0.004680,-3.000333,0.002697,-0.023213,-0.004869


In [ ]:
modeling.average_marginal_effects(res_onset_probit, modeling.EXISTENCE_TERMS)

,dy/dx,Std. Err.,z,Pr(>|z|),Conf. Int. Low,Cont. Int. Hi.
log_base_fee,-0.156073,0.056467,-2.763958,0.005710,-0.266747,-0.045399
gas_util_lag,-0.038573,0.012939,-2.981257,0.002871,-0.063933,-0.013214
tip_p90_lag,-0.000084,0.001939,-0.043387,0.965393,-0.003885,0.003716
mev_lag,-0.005328,0.006289,-0.847196,0.396886,-0.017654,0.006998
freq_lag,0.067257,0.020702,3.248770,0.001159,0.026681,0.107832
log_vol,0.273575,0.118283,2.312886,0.020729,0.041745,0.505406
dlogL_lag,-0.013336,0.004338,-3.074289,0.002110,-0.021839,-0.004834
